In [25]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
import function_sampling_Thermal_forcing_files as fn
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'


In [26]:
 
mnt_pth = '/Volumes/CrucialX8/computing/data/antarctica/' #mount path
pth_imip6hack= mnt_pth + 'ismip6_hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6_hackathon/ComputedScalarsHelene/'
 
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = [] # specify models to remove
 
metadata_file = 'Metadata.txt'
# Generate loop_info DataFrame
variable2 = 'groundingline_bmb'
y2sec=-365.25*24*3600
factor = y2sec/10**12
pth = pth_imip6hack

In [27]:
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]
dic_area_of_interest = {}
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']
dic_area_of_interest['Amundsen'] = ['_sector_4']


In [28]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    lst = lst[:-4]
    lst = lst[::-1]

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
  
    pth_tf = f'{pth}/{df.iloc[i].Experiment}/groundingline_corridor32/computed_{variable2}_corr_32_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'

       
    d3 = xr.open_dataset(pth_tf,decode_times=False )


    tmin = np.min([d3.time.size])
            #melting per area
        
   

    ind = np.where(d3.time.values <= 2100)
    
    
    
  
    nm = ''
    #melting per area
#     y =-1*d.shelfmelt.values[:tmin]* 365.25 * 24 * 3600/900/d2.iareafl.values[:tmin] #GT/m^3 yr
    yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
    
   

    yaisBMB =yBMB[ind].cumsum()[-1]
    

    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        nm = '_sector_'+str(j)
        yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
        
        
        mnan = np.isnan(yBMB)
        
        
       
       
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        nm = '_region_'+str(j)
        yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
        
       
        mnan = np.isnan(yBMB)
        
        
        
       
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
       
        for j,sector in enumerate(sectors):
            ys[j,:] = d3[f'{variable2}{sector}'].values[:tmin]*factor
        
           
        yBMB =  np.nansum(ys,axis = 0)
       
       
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_groundingline_melt_2100.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/150463659.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/150463659.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/150463659.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retu

In [29]:
df2

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,IMAU_UFEMISM4,/Volumes/CrucialX8/computing/data/antarctica/i...,1245.495166,149.328403,0.000000,22.185937,530.813222,79.711338,60.433004,...,25.566045,12.753704,0.000000,0.000000,0.000000,0.000000,825.214291,229.039742,22.185937,3.102777e+03
1,expAE02,IMAU_UFEMISM3,/Volumes/CrucialX8/computing/data/antarctica/i...,568.735901,97.845450,0.000000,779.019480,610.058884,92.125809,149.096132,...,7.839801,8.948574,0.000000,0.000000,0.000000,0.000000,982.106458,189.971258,779.019480,3.271295e+03
2,expAE02,IMAU_UFEMISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000e+00
3,expAE02,IMAU_UFEMISM1,/Volumes/CrucialX8/computing/data/antarctica/i...,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000e+00
4,expAE02,VUW_PISM2_s4,/Volumes/CrucialX8/computing/data/antarctica/i...,57533.847291,123634.720554,71436.263710,183288.462511,386698.756398,9271.739454,33479.539942,...,2971.638508,4956.735947,62548.125206,62548.125206,62548.125206,62548.125206,553611.392005,132906.460008,183288.462511,1.485348e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167,expAE05,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,16727.868726,100783.348442,43596.492656,68450.586852,117307.753315,26863.110194,42154.351803,...,4683.652864,7105.637466,31488.920669,31488.920669,31488.920669,31488.920669,191836.346110,127646.458636,68450.586852,7.942277e+05
168,expAE05,DOE_MALI_8km_AntMean,/Volumes/CrucialX8/computing/data/antarctica/i...,10240.249537,262482.169101,26731.386402,295898.314471,216629.055192,10944.565896,82704.085508,...,1355.590266,2140.085974,23540.066883,23540.066883,23540.066883,23540.066883,327135.920336,273426.734997,295898.314471,1.249078e+06
169,expAE05,DOE_MALI_8km_Ant95,/Volumes/CrucialX8/computing/data/antarctica/i...,10090.529171,221904.032325,28227.167433,273643.623442,188988.095991,9833.762748,73624.619539,...,1373.596861,2160.271375,23294.826989,23294.826989,23294.826989,23294.826989,288258.523250,231737.795073,273643.623442,1.135135e+06
170,expAE05,DOE_MALI_4km,/Volumes/CrucialX8/computing/data/antarctica/i...,13561.975361,270685.430108,33778.536557,245648.292529,232083.454915,26080.611071,85067.947480,...,2718.708926,3302.171211,27584.987527,27584.987527,27584.987527,27584.987527,354974.246306,296766.041179,245648.292529,1.313334e+06


In [30]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    lst = lst[:-4]
    lst = lst[::-1]

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
  
    pth_tf = f'{pth}/{df.iloc[i].Experiment}/groundingline_corridor32/computed_{variable2}_corr_32_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'

       
    d3 = xr.open_dataset(pth_tf,decode_times=False )


    tmin = np.min([d3.time.size])
            #melting per area
        
   

    ind = np.where(d3.time.values <= 2200)
    
    
    
  
    nm = ''
    #melting per area
#     y =-1*d.shelfmelt.values[:tmin]* 365.25 * 24 * 3600/900/d2.iareafl.values[:tmin] #GT/m^3 yr
    yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
    
   

    yaisBMB =yBMB[ind].cumsum()[-1]
    

    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        nm = '_sector_'+str(j)
        yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
        
        
        mnan = np.isnan(yBMB)
        
        
       
       
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        nm = '_region_'+str(j)
        yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
        
       
        mnan = np.isnan(yBMB)
        
        
        
       
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
       
        for j,sector in enumerate(sectors):
            ys[j,:] = d3[f'{variable2}{sector}'].values[:tmin]*factor
        
           
        yBMB =  np.nansum(ys,axis = 0)
       
       
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_groundingline_melt_2200.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/1648664414.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/1648664414.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/1648664414.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#r

In [31]:
# avg and abs 2100
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_helene+f'{exp}/shelfmelt/*.nc')
    lst.sort()
    lst = lst[:-4]
    lst = lst[::-1]

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
for i in range(1,4):
    df['region_'+str(i)]=0
for key in dic_area_of_interest.keys():
    df[key]=0
df['AIS']=0
df2 = df.copy()
for i in range (df.shape[0]):
  
    pth_tf = f'{pth}/{df.iloc[i].Experiment}/groundingline_corridor32/computed_{variable2}_corr_32_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'

       
    d3 = xr.open_dataset(pth_tf,decode_times=False )


    tmin = np.min([d3.time.size])
            #melting per area
        
   

    ind = np.where(d3.time.values <= 2300)
    
    
    
  
    nm = ''
    #melting per area
#     y =-1*d.shelfmelt.values[:tmin]* 365.25 * 24 * 3600/900/d2.iareafl.values[:tmin] #GT/m^3 yr
    yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
    
   

    yaisBMB =yBMB[ind].cumsum()[-1]
    

    df2['AIS'].iloc[i]=  yaisBMB
    

    for j in range(1,19):
        nm = '_sector_'+str(j)
        yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
        
        
        mnan = np.isnan(yBMB)
        
        
       
       
        df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
    for j in range(1,4):
        nm = '_region_'+str(j)
        yBMB =d3[f'{variable2}{nm}'].values[:tmin]*factor
        
       
        mnan = np.isnan(yBMB)
        
        
        
       
        df2['region_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
        
    for key in dic_area_of_interest.keys():
        sectors = dic_area_of_interest[key].copy()
        ys = np.zeros((len(sectors),tmin))
       
        for j,sector in enumerate(sectors):
            ys[j,:] = d3[f'{variable2}{sector}'].values[:tmin]*factor
        
           
        yBMB =  np.nansum(ys,axis = 0)
       
       
        df2[key].iloc[i] = yBMB[ind].cumsum()[-1]
        
        
#         if (j==1 and i ==4):
#             print(d2['iareafl_sector_'+str(j)].values[:tmin])
#             break
# df.to_csv('tables/averegeshelfmelt_2100.csv', index=False)
df2.to_csv('tables/cumulative_groundingline_melt_2300.csv', index=False)

/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/1063978827.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['AIS'].iloc[i]=  yaisBMB
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/1063978827.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['sector_'+str(j)].iloc[i] = yBMB[ind].cumsum()[-1]
/var/folders/ph/m61jd3r90t190p6qvbps3yqct2hr65/T/ipykernel_33601/1063978827.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#r

In [18]:
 df2

,Experiment,Model,Path,sector_1,sector_2,sector_3,sector_4,sector_5,sector_6,sector_7,...,sector_16,sector_17,sector_18,region_1,region_2,region_3,FilchnerRonne,Ross,Amundsen,AIS
0,expAE02,DC_ISSM,/Volumes/CrucialX8/computing/data/antarctica/i...,3.362330e+13,6.398796e+13,5.018167e+13,8.539315e+13,1.430541e+14,1.725435e+13,2.218547e+13,...,5.054789e+12,8.742452e+12,4.590918e+13,4.590918e+13,4.590918e+13,4.590918e+13,2.503479e+14,8.124230e+13,8.539315e+13,8.966398e+14
1,expAE02,DOE_MALI_4km,/Volumes/CrucialX8/computing/data/antarctica/i...,2.001348e+13,8.301372e+13,4.755132e+13,1.591927e+14,1.951389e+14,1.714101e+13,3.147759e+13,...,4.348871e+12,7.688526e+12,3.418665e+13,3.418665e+13,3.418665e+13,3.418665e+13,3.103140e+14,1.001547e+14,1.591927e+14,9.873332e+14
2,expAE02,DOE_MALI_8km_Ant95,/Volumes/CrucialX8/computing/data/antarctica/i...,1.775872e+13,6.531180e+13,4.202989e+13,2.033690e+14,1.428459e+14,6.801653e+12,1.976042e+13,...,2.276318e+12,5.585600e+12,2.928723e+13,2.928723e+13,2.928723e+13,2.928723e+13,2.361469e+14,7.211346e+13,2.033690e+14,8.523569e+14
3,expAE02,DOE_MALI_8km_AntMean,/Volumes/CrucialX8/computing/data/antarctica/i...,1.853268e+13,8.263261e+13,4.190862e+13,2.429890e+14,1.636955e+14,7.567192e+12,3.001640e+13,...,2.420147e+12,5.590800e+12,2.881733e+13,2.881733e+13,2.881733e+13,2.881733e+13,2.658979e+14,9.019980e+13,2.429890e+14,9.606228e+14
4,expAE02,IGE_ElmerIce,/Volumes/CrucialX8/computing/data/antarctica/i...,2.162735e+13,5.590985e+13,4.820469e+13,6.004382e+13,1.268319e+14,2.135368e+13,2.765592e+13,...,5.591984e+12,9.446197e+12,3.856016e+13,3.856016e+13,3.856016e+13,3.856016e+13,2.138862e+14,7.726353e+13,6.004382e+13,7.827219e+14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,expAE05,IMAU_UFEMISM4,/Volumes/CrucialX8/computing/data/antarctica/i...,2.016237e+12,1.371665e+11,-0.000000e+00,2.154112e+10,4.858696e+11,7.371025e+10,1.120927e+11,...,5.611791e+09,1.170916e+10,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,8.147092e+11,2.108768e+11,2.154112e+10,3.517450e+12
184,expAE05,IMAU_UFEMISM1,/Volumes/CrucialX8/computing/data/antarctica/i...,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,...,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-0.000000e+00
185,expAE05,IMAU_UFEMISM2,/Volumes/CrucialX8/computing/data/antarctica/i...,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,...,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-0.000000e+00
186,expAE05,IMAU_UFEMISM3,/Volumes/CrucialX8/computing/data/antarctica/i...,3.431461e+11,2.188149e+11,-0.000000e+00,1.336076e+12,5.037408e+11,8.977977e+10,2.974978e+11,...,8.878997e+09,3.734432e+09,-0.000000e+00,-0.000000e+00,-0.000000e+00,-0.000000e+00,8.634442e+11,3.085946e+11,1.336076e+12,3.934423e+12
